<a href="https://colab.research.google.com/github/aabha25/nexus/blob/main/XGBOOST_Final50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

In [9]:
# df = pd.read_csv('/content/drive/MyDrive/Flattened_Input_XGBoost_WithAgeSexDecline_16-8-25.csv')
# df.shape
df = pd.read_csv('/content/drive/MyDrive/trainxgboost.csv')
df.shape

(305, 399)

In [10]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,389,390,391,392,393,394,395,396,397,398
0,75,0,30.728432,0.0,7.730588,5.025456,1.823457,0.0,9.790758,0.0,...,3.256823,0.0,0.0,6.174289,24.584297,0.0,0.0,8.00,16.00,2
1,83,0,33.105865,0.0,14.344643,5.134811,4.778924,0.0,10.616479,0.0,...,0.000000,0.0,0.0,11.245796,24.674185,0.0,0.0,4.67,7.67,2
2,79,1,46.937030,0.0,19.009192,3.151681,6.104017,0.0,18.568846,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,76,0,40.971775,0.0,15.296716,5.286087,5.741526,0.0,14.200731,0.0,...,0.000000,0.0,0.0,12.689952,25.582218,0.0,0.0,4.00,6.00,2
4,86,0,33.429400,0.0,13.165087,4.160962,5.787603,0.0,10.437432,0.0,...,2.256289,0.0,0.0,10.187354,25.039909,0.0,0.0,2.00,4.00,2


In [6]:
#increamental cha random search

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# Prepare your data
X = df.iloc[:, :398].replace(-1, np.nan).values
y = df.iloc[:, 398]

# param_grid = {
#     'n_estimators': [10, 20, 50],  # low because we'll train incrementally
#     'max_depth': [3, 5, 7],
#     'learning_rate': [0.01, 0.05, 0.1],
#     'subsample': [0.7, 1.0],
#     'colsample_bytree': [0.7, 1.0],
#     'gamma': [0, 0.1],
#     'min_child_weight': [1, 3],
#     'reg_alpha': [0, 0.1],
#     'reg_lambda': [1, 2]
# }

param_grid = {
    'n_estimators': [50, 100, 200],          # Increase for more robust trees even incrementally
    'max_depth': [3, 5, 7, 9],               # Add deeper trees if data is complex
    'learning_rate': [0.005, 0.01, 0.05, 0.1], # Include slower rates for finer updates
    'subsample': [0.5, 0.7, 0.9, 1.0],       # More variability in sampling data rows
    'colsample_bytree': [0.5, 0.7, 0.9, 1.0],# More variability in sampling features
    'gamma': [0, 0.1, 0.3, 0.5],             # Try stronger pruning options
    'min_child_weight': [1, 3, 5, 7],        # Regularize deeper splits
    'reg_alpha': [0, 0.1, 0.5, 1],           # L1 regularization to avoid overfitting
    'reg_lambda': [1, 1.5, 2, 5]             # L2 regularization
}


model = xgb.XGBClassifier(
    eval_metric='mlogloss',
    use_label_encoder=False
)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=30,
    scoring='accuracy',
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X, y)

best_params = random_search.best_params_
print("Best hyperparameters:", best_params)


Fitting 3 folds for each of 30 candidates, totalling 90 fits


/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [17:57:31] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best hyperparameters: {'subsample': 0.7, 'reg_lambda': 2, 'reg_alpha': 1, 'n_estimators': 50, 'min_child_weight': 7, 'max_depth': 7, 'learning_rate': 0.005, 'gamma': 0.5, 'colsample_bytree': 0.9}


In [11]:
#increamental model

# # batch_size = 100
# n_samples = X.shape[0]
# n_batches = n_samples // batch_size
X = df.iloc[:, :398].replace(-1, np.nan).values
y = df.iloc[:, 398]
# Initialize model with best hyperparameters
incremental_model = xgb.XGBClassifier(
    **best_params,
    eval_metric='mlogloss',
    use_label_encoder=False
)

prev_model = None

for i in range(1,7):
     X_test = np.full((305, 398), np.nan)
     X_test[:, :66*i+2] = X[:, :66*i+2]  # <-- take first 66 * i columns from X
     y = df.iloc[:, 398]



     incremental_model.fit(
        X_test, y,
        xgb_model=prev_model
     )

     prev_model = incremental_model.get_booster()


/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:08:35] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:08:36] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:08:37] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:08:38] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:08:39] WARNING: /w

In [12]:

def mask_all_after(X, n_unmasked=66):
    X_masked = X.copy()
    X_masked[:, n_unmasked:] = np.nan
    return X_masked

# Masked data: only first 66 features present
for i in range(1,7):
  X_masked = mask_all_after(X, n_unmasked=66*i)

  # Continue training the existing best_model on masked data
  incremental_model.fit(
      X_masked, y,
      xgb_model=incremental_model.get_booster()
  )

/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:08:58] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:09:00] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:09:01] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:09:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:09:03] WARNING: /w

In [ ]:
# for j in range(3):
#   for i in range(1, 7):
#     X_masked = mask_all_after(X, n_unmasked=66*i)

#     # Continue training the existing best_model on masked data
#     incremental_model.fit(
#         X_masked, y,
#         xgb_model=incremental_model.get_booster()
#     )

In [26]:
# for j in range(3):
X = df.iloc[:, :398].replace(-1, np.nan).values
y = df.iloc[:, 398]
for i in range(1, 7):
    X_masked = mask_all_after(X, n_unmasked=66*i)

    # Continue training the existing best_model on masked data
    incremental_model.fit(
        X_masked, y,
        xgb_model=incremental_model.get_booster()
    )

/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:18:05] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:18:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:18:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:18:09] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [18:18:12] WARNING: /w

In [27]:
from sklearn.metrics import accuracy_score

for i in range(1, 7):
    X_test = np.full((305, 398), np.nan)
    X_test[:, :66*i+2] = X[:, :66*i+2]
    preds = incremental_model.predict(X_test)
    acc = accuracy_score(y[:], preds)
    print(f"Accuracy with first {66*i} features: {acc}")

Accuracy with first 66 features: 0.6
Accuracy with first 132 features: 0.8819672131147541
Accuracy with first 198 features: 0.9147540983606557
Accuracy with first 264 features: 0.9442622950819672
Accuracy with first 330 features: 0.9639344262295082
Accuracy with first 396 features: 0.9704918032786886


In [ ]:
#TESTING

In [21]:
dftest = pd.read_csv('/content/drive/MyDrive/testxgboost.csv')
dftest.shape

(77, 399)

In [28]:
Xt = dftest.iloc[:, :398].replace(-1, np.nan).values
yt = dftest.iloc[:, 398]

from sklearn.metrics import accuracy_score

for i in range(1, 7):
    X_test = np.full((77, 398), np.nan)
    X_test[:, :66*i+2] = Xt[:, :66*i+2]
    preds = incremental_model.predict(X_test)
    acc = accuracy_score(yt[:], preds)
    print(f"Accuracy with first {66*i} features: {acc}")

Accuracy with first 66 features: 0.5714285714285714
Accuracy with first 132 features: 0.6623376623376623
Accuracy with first 198 features: 0.6753246753246753
Accuracy with first 264 features: 0.6623376623376623
Accuracy with first 330 features: 0.6883116883116883
Accuracy with first 396 features: 0.6753246753246753
